# Week 3 Exercise — Game Content / Design Data Generator

**Week 3 task:** Create your own tool that generates synthetic data. Input the type of dataset (or products, job postings, etc.) and let the tool produce sample data.

**This solution:** A **game content generator** for indie/solo devs. Choose a content type (abilities, weapons, items, achievements, quests, etc.), add a short description or genre, and generate a JSON array of synthetic design data you can use for prototyping, content pipelines, or test data.

## Why this tool matters

Solo and small teams often need **lots of varied content** (weapons, abilities, quests, items) without the time to hand-write every entry. This generator gives you **synthetic design data** in a consistent JSON shape so you can:

- **Prototype fast** — drop the JSON into your game or editor and iterate on systems.
- **Test UIs and pipelines** — e.g. inventory, loot tables, achievement screens.
- **Kickstart a content doc** — use the output as a first draft and then edit.

**Example:** You’re making a **fantasy RPG** and need a **variety of weapons**. Select **Weapons** as the content type, fill in the description (e.g. *"Fantasy RPG: swords, axes, bows, staves. Include damage type and a short flavour line."*), choose how many samples you want, and click **Generate**. You get a JSON array of weapon entries (name, type, damage, description, rarity) that you can plug straight into your game data or design doc.

In [1]:
# Imports
import os
import json
import re
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Setup
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

client = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key)

MODELS = {
    "gpt-4o-mini": "openai/gpt-4o-mini",
    "Llama": "ollama/llama3.2",
}
DEFAULT_MODEL = "gpt-4o-mini"

In [3]:
# Content types: each has a label and a schema hint for the model so output is consistent
CONTENT_TYPES = {
    "Abilities / skills": "Each object: name, description, effect (what it does), cooldown_or_cost (e.g. '8s' or '20 mana'), rarity (common/rare/epic).",
    "Weapons": "Each object: name, weapon_type (e.g. sword, bow), damage_or_effect, description, rarity.",
    "Items / consumables": "Each object: name, description, effect, consumable (true/false), rarity.",
    "Achievements": "Each object: id, title, description, criteria (how to unlock), reward (optional, e.g. '10 gold').",
    "Quests": "Each object: id, title, description, objectives (array of short strings), reward.",
    "NPCs / characters": "Each object: name, role, one_line, dialogue_hook (short sample line).",
    "Enemies": "Each object: name, type, health_range (e.g. '50-80'), behavior_notes, loot_notes.",
}

def get_system_prompt(content_type: str, num_samples: int) -> str:
    schema_hint = CONTENT_TYPES.get(content_type, "Each object: name, description, and any other relevant fields.")
    return f"""You are a game design assistant. Generate exactly {num_samples} synthetic game content entries.

Content type: {content_type}
Schema (use these field names): {schema_hint}

Output valid JSON only: a single object with one key "items" (or "data") whose value is an array of {num_samples} objects. No markdown, no explanation. Example shape: {{ "items": [ {{ "name": "...", ... }}, ... ] }}"""

In [4]:
def extract_json_array(text: str):
    """Get a JSON array from model output (handles wrapper object or raw array)."""
    if not text or not text.strip():
        raise ValueError("Empty model output.")
    t = text.strip()
    # Remove markdown code fence if present
    if "```" in t:
        t = re.sub(r"^```(?:json)?\s*|\s*```$", "", t, flags=re.IGNORECASE | re.MULTILINE).strip()
    try:
        obj = json.loads(t)
    except json.JSONDecodeError:
        start, end = t.find("["), t.rfind("]")
        if start != -1 and end != -1 and end > start:
            t = t[start : end + 1]
            t = re.sub(r",\s*([\]}])", r"\1", t)  # trailing commas
            obj = json.loads(t)
        else:
            raise ValueError("No JSON array found in output.")
    if isinstance(obj, list):
        return obj
    for key in ("items", "data", "entries", "results"):
        if isinstance(obj.get(key), list):
            return obj[key]
    raise ValueError("JSON object has no array field (items/data/entries/results).")

def generate_game_content(content_type: str, description: str, num_samples: int, model_name: str) -> str:
    """Generate synthetic game content and return formatted JSON string."""
    num_samples = max(1, min(20, int(num_samples)))
    system = get_system_prompt(content_type, num_samples)
    user = description.strip() if description else "Generate varied, creative entries suitable for a solo or small-team game."
    model_id = MODELS.get(model_name, MODELS[DEFAULT_MODEL])
    try:
        resp = client.chat.completions.create(
            model=model_id,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
            response_format={"type": "json_object"},
        )
        raw = resp.choices[0].message.content or ""
        arr = extract_json_array(raw)
        return json.dumps(arr, indent=2)
    except Exception as e:
        return json.dumps({"error": str(e)}, indent=2)

In [5]:
# Gradio UI
def run_generate(content_type, description, num_samples, model_name):
    if not content_type:
        return json.dumps({"error": "Pick a content type."}, indent=2)
    return generate_game_content(content_type, description, num_samples, model_name)

with gr.Blocks(title="Game Content Generator") as app:
    gr.Markdown("Generate synthetic game design data. Choose a content type, add optional description (e.g. genre or theme), and click **Generate**.")
    content_dropdown = gr.Dropdown(
        choices=list(CONTENT_TYPES.keys()),
        value=list(CONTENT_TYPES.keys())[0],
        label="Content type",
    )
    description_box = gr.Textbox(
        label="Description (optional)",
        placeholder="e.g. roguelike abilities, fantasy RPG weapons, sci-fi quests",
        lines=2,
    )
    with gr.Row():
        num_slider = gr.Slider(1, 20, value=5, step=1, label="Number of samples")
        model_dropdown = gr.Dropdown(choices=list(MODELS.keys()), value=DEFAULT_MODEL, label="Model")
    generate_btn = gr.Button("Generate", variant="primary")
    output_json = gr.Code(label="Generated JSON", language="json", lines=20)

    generate_btn.click(
        run_generate,
        [content_dropdown, description_box, num_slider, model_dropdown],
        output_json,
    )

In [ ]:
app.launch()